In [0]:
%sql
CREATE TABLE IF NOT EXISTS
bank_catalog.bronze.batch_audit
(
    batch_id STRING,
    table_name STRING,
    status STRING,
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    records_processed BIGINT,
    error_message STRING
)
USING DELTA;

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
source_path = "/Volumes/bank_catalog/bronze/raw_files/databricks_bank_batch_data"

tables = [
    "customers",
    "accounts",
    "transactions",
    "loans",
    "branches"
]

primary_keys = {
    "customers": "customer_id",
    "accounts": "account_id",
    "transactions": "transaction_id",
    "loans": "loan_id",
    "branches": "branch_id"
}

In [0]:
def get_batch_status(batch_id, table_name):

    audit = spark.table(
        "bank_catalog.bronze.batch_audit"
    )

    result = (
        audit.filter(
            (F.col("batch_id") == batch_id) &
            (F.col("table_name") == table_name) &
            (F.col("status") == "SUCCESS")
        )
        .limit(1)
        .count()
    )

    return result > 0

In [0]:
audit_columns = {
    row.col_name
    for row in spark.sql(
        "DESCRIBE bank_catalog.bronze.batch_audit"
    ).collect()
    if row.col_name and not row.col_name.startswith("#")
}

if "event_timestamp" not in audit_columns:
    spark.sql(
        "ALTER TABLE bank_catalog.bronze.batch_audit "
        "ADD COLUMNS (event_timestamp TIMESTAMP)"
    )

In [0]:
def log_batch(batch_id, table_name, status,
              records=0, error=None):

    audit_df = spark.createDataFrame(
        [(batch_id, table_name, status,
          records, error)],
        "batch_id STRING, table_name STRING, status STRING, "
        "records_processed BIGINT, error_message STRING"
    )

    audit_df = audit_df.withColumn(
        "event_timestamp", F.current_timestamp()
    )

    audit_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable("bank_catalog.bronze.batch_audit")

In [0]:
def ingest_batch(batch_id):

    for table in tables:

        try:

            # Check whether batch is already completed
            if get_batch_status(batch_id, table):

                print(f"Skipping {batch_id} - {table}")
                continue

            print(f"Processing {batch_id} - {table}")

            log_batch(
                batch_id, table, "STARTED"
            )

            file_path = (
                f"{source_path}/{batch_id}/{table}_raw.csv"
            )

            # Read source CSV
            df = (
                spark.read
                .format("csv")
                .option("header", "true")
                .option("inferSchema", "false")
                .load(file_path)
            )

            # Add metadata
            df = (
                df.withColumn("batch_id", F.lit(batch_id))
                  .withColumn(
                      "source_file",
                      F.lit(f"{table}_raw.csv")
                  )
                  .withColumn(
                      "ingestion_timestamp",
                      F.current_timestamp()
                  )
            )

            target_table = f"bank_catalog.bronze.{table}"

            primary_key = primary_keys[table]

            if not spark.catalog.tableExists(target_table):

                df.write \
                    .format("delta") \
                    .mode("overwrite") \
                    .saveAsTable(target_table)

            else:

                delta_table = DeltaTable.forName(
                    spark, target_table
                )

                condition = (
                    f"target.batch_id = source.batch_id "
                    f"AND target.{primary_key} = source.{primary_key}"
                )

                (
                    delta_table.alias("target")
                    .merge(df.alias("source"), condition)
                    .whenMatchedUpdateAll()
                    .whenNotMatchedInsertAll()
                    .execute()
                )

            log_batch(
                batch_id,
                table,
                "SUCCESS",
                df.count()
            )

            print(f"Successfully completed {batch_id} - {table}")

        except Exception as e:

            log_batch(
                batch_id,
                table,
                "FAILED",
                0,
                str(e)[:2000]
            )

            print(f"Failed {batch_id} - {table}: {e}")
            raise

In [0]:
ingest_batch("batch_002")

ingest_batch("batch_003")